# Qwen3-TTS Text-to-Speech with OpenVINO™

Qwen3-TTS is a state-of-the-art text-to-speech model developed by Alibaba's Qwen team. It offers high-quality speech synthesis with support for multiple languages, voice cloning, and voice design features.
Qwen3-TTS covers 10 major languages (Chinese, English, Japanese, Korean, German, French, Russian, Portuguese, Spanish, and Italian) as well as multiple dialectal voice profiles to meet global application needs. In addition, the models feature strong contextual understanding, enabling adaptive control of tone, speaking rate, and emotional expression based on instructions and text semantics, and they show markedly improved robustness to noisy input text. Key features:

* **Powerful Speech Representation**: Powered by the self-developed Qwen3-TTS-Tokenizer-12Hz, it achieves efficient acoustic compression and high-dimensional semantic modeling of speech signals. It fully preserves paralinguistic information and acoustic environmental features, enabling high-speed, high-fidelity speech reconstruction through a lightweight non-DiT architecture.
* **Universal End-to-End Architecture**: Utilizing a discrete multi-codebook LM architecture, it realizes full-information end-to-end speech modeling. This completely bypasses the information bottlenecks and cascading errors inherent in traditional LM+DiT schemes, significantly enhancing the model’s versatility, generation efficiency, and performance ceiling.
* **Extreme Low-Latency Streaming Generation**: Based on the innovative Dual-Track hybrid streaming generation architecture, a single model supports both streaming and non-streaming generation. It can output the first audio packet immediately after a single character is input, with end-to-end synthesis latency as low as 97ms, meeting the rigorous demands of real-time interactive scenarios.
* **Intelligent Text Understanding and Voice Control**: Supports speech generation driven by natural language instructions, allowing for flexible control over multi-dimensional acoustic attributes such as timbre, emotion, and prosody. By deeply integrating text semantic understanding, the model adaptively adjusts tone, rhythm, and emotional expression, achieving lifelike “what you imagine is what you hear” output.

<p align="center">
    <img src="https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-TTS-Repo/overview.png" width="80%"/>
<p>
    
More details can be found in the original [repository](https://github.com/QwenLM/Qwen3-TTS) and [model card](https://huggingface.co/collections/Qwen/qwen3-tts-6840560df53539afc9a0401b)

In this tutorial, we will:
1. Install required dependencies
2. Download and convert Qwen3-TTS model to OpenVINO format
3. Create an inference pipeline using OpenVINO
4. Build an interactive Gradio demo for text-to-speech

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Download and Prepare Qwen3-TTS Model](#Download-and-Prepare-Qwen3-TTS-Model)
- [Convert Model to OpenVINO Format](#Convert-Model-to-OpenVINO-Format)
- [Create Inference Pipeline](#Create-Inference-Pipeline)
    - [Select Inference Device](#Select-Inference-Device)
    - [Run Text-to-Speech](#Run-Text-to-Speech)
- [Interactive Demo](#Interactive-Demo)


⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO. It may be fully supported and validated in the future.


<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/qwen3-tts/qwen3-tts.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
# Fetch notebook_utils module
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen3-tts.ipynb")

### Install Dependencies
[back to top ⬆️](#Table-of-contents:)

Install required packages for Qwen3-TTS model conversion and inference:

In [ ]:
from cmd_helper import clone_repo
from pip_helper import pip_install
import platform

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch==2.8.0",
    "nncf",
    "torchaudio==2.8.0",
    "openvino>=2025.4.0",
    "gradio>=4.0",
    "huggingface_hub",
)

# Clone Qwen3-TTS repository for model code
repo_dir = Path("Qwen3-TTS")
revision = "1ab0dd75353392f28a0d05d9ca960c9954b13c83"
clone_repo("https://github.com/QwenLM/Qwen3-TTS.git", revision)

pip_install(
    "-q -e",
    str(repo_dir),
)

if platform.system() == "Darwin":
    pip_install("numpy<2.0")

## Download and Prepare Qwen3-TTS Model
[back to top ⬆️](#Table-of-contents:)

Qwen3-TTS offers different model variants:

| Model Type | Size | Description |
|------------|------|-------------|
| **CustomVoice** | 0.6B / 1.7B | Use predefined speaker voices with optional style instructions |
| **Base** | 0.6B / 1.7B | Clone any voice from reference audio |
| **VoiceDesign** | 1.7B | Create custom voices using natural language descriptions |

Select your preferred model:

In [3]:
import ipywidgets as widgets

model_options = {
    "Qwen3-TTS-CustomVoice-0.6B": "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",
    "Qwen3-TTS-CustomVoice-1.7B": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
    "Qwen3-TTS-Base-0.6B": "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
    "Qwen3-TTS-Base-1.7B": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    "Qwen3-TTS-VoiceDesign-1.7B": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
}

model_selector = widgets.Dropdown(
    options=list(model_options.keys()),
    value="Qwen3-TTS-CustomVoice-0.6B",
    description="Model:",
    style={"description_width": "initial"},
)

model_selector

Dropdown(description='Model:', options=('Qwen3-TTS-CustomVoice-0.6B', 'Qwen3-TTS-CustomVoice-1.7B', 'Qwen3-TTS…

### Model Architecture

The Qwen3-TTS pipeline consists of several key components:

* **Text Embeddings**: Converts text tokens into embeddings for the language model
* **Talker Language Model**: A decoder-only transformer that generates speech codes from text
* **Code Predictor**: Predicts additional speech codes for high-quality synthesis
* **Speech Tokenizer**: Converts speech codes to audio waveform
* **Speaker Encoder** (Base model only): Extracts speaker embeddings from reference audio for voice cloning

The model uses a multi-stage approach: first, the Talker generates discrete speech codes conditioned on text and speaker information, then the Speech Tokenizer decodes these codes into natural speech audio.

## Convert Model to OpenVINO Format
[back to top ⬆️](#Table-of-contents:)

Now we'll convert the Qwen3-TTS model to OpenVINO Intermediate Representation (IR) format. The conversion process exports the following components:

1. **Talker Embedding Model** (`openvino_talker_embedding.xml`): Codec token embedding layer
2. **Talker Text Embedding Model** (`openvino_talker_text_embedding.xml`): Text token embedding layer
3. **Talker Text Projection Model** (`openvino_talker_text_projection.xml`): Projects text embeddings
4. **Talker Language Model** (`openvino_talker_language_model.xml`): The main decoder with KV-cache support
5. **Code Predictor Embedding Model** (`openvino_talker_code_predictor_embedding.xml`): Code predictor embedding
6. **Code Predictor Model** (`openvino_talker_code_predictor.xml`): Predicts additional speech codes
7. **Speaker Encoder** (Base only) (`openvino_speaker_encoder.xml`): Extracts speaker embeddings

In [4]:
from qwen_3_tts_helper import convert_qwen3_tts_model

model_name = model_selector.value
model_id = model_options[model_name]
ov_model_dir = Path(f"{model_name}-OV")

# Convert model to OpenVINO format
# This will skip conversion if the model already exists
convert_qwen3_tts_model(
    model_id=model_id,
    output_dir=ov_model_dir,
    quantization_config=None,  # Set to {"mode": "INT8_SYM"} for INT8 quantization
)

✅ Patched torch.diff in masking_utils.find_packed_sequence_indices for OpenVINO compatibility


SoX could not be found!

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    
/bin/sh: 1: sox: Permission denied



********
********
 
✅ Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice model already converted. You can find results in Qwen3-TTS-CustomVoice-0.6B-OV


## Create Inference Pipeline
[back to top ⬆️](#Table-of-contents:)

### Select Inference Device
[back to top ⬆️](#Table-of-contents:)

Select the device for running inference. CPU provides the most compatibility, while GPU and NPU can provide acceleration on supported hardware:

In [ ]:
from notebook_utils import device_widget

device = device_widget("CPU", exclude=["NPU"])

device

Dropdown(description='Device:', options=('CPU', 'GPU'), value='CPU')

### Load OpenVINO Model
[back to top ⬆️](#Table-of-contents:)

Load the converted OpenVINO model using our helper class. The `OVQwen3TTSModel` provides the same API as the original Qwen3TTSModel:

In [6]:
from qwen_3_tts_helper import OVQwen3TTSModel

ov_model = OVQwen3TTSModel.from_pretrained(
    model_dir=ov_model_dir,
    device=device.value,
)

print("✓ Model loaded successfully!")
print(f"  Model type: {ov_model.tts_model_type}")
print(f"  Model size: {ov_model.tts_model_size}")
print(f"  Supported speakers: {ov_model.get_supported_speakers()}")
print(f"  Supported languages: {ov_model.get_supported_languages()}")

No checkpoint_path.txt found, trying to load processor from model_dir
Loading OpenVINO speech tokenizer
✓ Model loaded successfully!
  Model type: custom_voice
  Model size: 0b6
  Supported speakers: ['aiden', 'dylan', 'eric', 'ono_anna', 'ryan', 'serena', 'sohee', 'uncle_fu', 'vivian']
  Supported languages: ['auto', 'chinese', 'english', 'french', 'german', 'italian', 'japanese', 'korean', 'portuguese', 'russian', 'spanish']


### Run Text-to-Speech
[back to top ⬆️](#Table-of-contents:)

Now let's test the model with some sample text. The usage depends on the model type:

- **CustomVoice**: Use predefined speakers with optional style instructions
- **Base**: Clone voice from reference audio
- **VoiceDesign**: Create custom voice with natural language description

In [7]:
import time
import IPython.display as ipd

# Test text
test_text = "Hello! Welcome to the Qwen3-TTS demo with OpenVINO acceleration. This is a test of text-to-speech synthesis."

print(f"Model type: {ov_model.tts_model_type}")
print(f"Input text: {test_text}")
print("\nGenerating speech...")

start_time = time.time()

if ov_model.tts_model_type == "custom_voice":
    # CustomVoice model - use predefined speaker
    wavs, sr = ov_model.generate_custom_voice(
        text=test_text,
        speaker="ryan",  # Available: aiden, dylan, eric, ono_anna, ryan, serena, sohee, uncle_fu, vivian
        language="English",
        instruct="Speak in a friendly and professional tone.",
        max_new_tokens=2048,
    )
elif ov_model.tts_model_type == "voice_design":
    # VoiceDesign model - create custom voice with description
    wavs, sr = ov_model.generate_voice_design(
        text=test_text,
        language="English",
        instruct="A warm, friendly male voice with a slight accent.",
        max_new_tokens=2048,
    )
elif ov_model.tts_model_type == "base":
    # Base model - needs reference audio for voice cloning
    # For demo, we'll use x_vector_only_mode without reference audio
    print("Note: Base model requires reference audio for full voice cloning.")
    print("Using x_vector_only_mode for demo...")
    # This is a placeholder - you would normally provide ref_audio
    wavs, sr = None, None
    print("Please provide reference audio for voice cloning.")

inference_time = time.time() - start_time

if wavs is not None:
    audio_duration = len(wavs[0]) / sr
    print("\n✓ Generation completed!")
    print(f"  Inference time: {inference_time:.2f}s")
    print(f"  Audio duration: {audio_duration:.2f}s")
    print(f"  Real-time factor (RTF): {inference_time/audio_duration:.3f}")
    print(f"  Sample rate: {sr} Hz")

    # Play audio
    ipd.display(ipd.Audio(wavs[0], rate=sr))

Model type: custom_voice
Input text: Hello! Welcome to the Qwen3-TTS demo with OpenVINO acceleration. This is a test of text-to-speech synthesis.

Generating speech...

✓ Generation completed!
  Inference time: 9.48s
  Audio duration: 7.98s
  Real-time factor (RTF): 1.189
  Sample rate: 24000 Hz


## Interactive Demo
[back to top ⬆️](#Table-of-contents:)

Launch an interactive Gradio demo for text-to-speech synthesis:

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_model, model_type=ov_model.tts_model_type)

# if you are launching remotely, specify server_name and server_port
#  demo.launch(server_name='your server name', server_port='server port in int')
# if you have any issue to launch on your platform, you can pass share=True to launch method:
# demo.launch(share=True)
# it creates a publicly shareable link for the interface. Read more in the docs: https://gradio.app/docs/
try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)